In this module, we'll be looking at **Reshaping and Stacking Arrays**. This skill will help me to *bend and combine NumPy arrays* — something that comes up constantly in data pipelines and quant workflows (e.g: reshaping price matrices, stacking multiple assets' return series, feeding data into ML models).<br><br>

**Why does this actually matter? A real-life example:**<br>
You pull daily returns for 5 stocks over 252 trading days. Your data might arrive as a flat 1D array of 1,260 numbers. To run any matrix operation (correlations, portfolio weights, covariance matrices), you need to reshape it into a (252, 5) grid — one row per day, one column per stock...

**Part 1 - Reshaping**<br>
*Making use of reshape()*<br><br>
reshape() changes the shape of an array without changing its data. Think of it like rearranging the same tiles into a different grid.

In [1]:
import numpy as np

In [2]:
#Rmagine 12 daily returns (flat, 1D)...
print("My Array:")
returns = np.arange(1,13) #This will produce a range between 1 and 13 exclusive of the last number
returns2 = np.array([[1,2],[3,4]])
print(returns)
print("Shape: ",returns.shape) #(row elements, columns)
# print("")
# print(returns2)
# print(returns2.shape) #(row elements, columns)

print("\nReshaping using .reshape():")
#Reshaping into 3 time periods x 4 weeks
matrix = returns.reshape(3,4) # I want 3 rows of 4 columns
print(matrix)
print("New Shape: ",matrix.shape) #(row elements, columns)



My Array:
[ 1  2  3  4  5  6  7  8  9 10 11 12]
Shape:  (12,)

Reshaping using .reshape():
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
New Shape:  (3, 4)


**Note**<br>
🔑 The Golden Rule of Reshape

Rows × Columns must equal total elements.
3 × 4 = 12 ✅ → works
3 × 5 = 15 ❌ → ValueError

In [26]:
#THESE SHOULDN"T WORK (SEE NOTE ABOVE)
matrix2 = returns.reshape(3,5) # I want 3 rows of 5 columns - not enough data!
print(matrix2)

matrix3 = returns.reshape(5,3) # I want 5 rows of 3 columns - not enough data!
print(matrix3)

matrix4 = returns.reshape(3,2) # I want 2 rows of 3 columns - not enough data!
print(matrix4)

ValueError: cannot reshape array of size 12 into shape (3,5)

🪄 The -1 Wildcard — Let NumPy Do the Maths

If you know one dimension but not the other, use -1 and NumPy figures it out:

In [45]:
returns3 = np.arange(1, 13)

# "I want 4 columns — you work out the rows"
matrix5 = returns3.reshape(-1, 4)
print(matrix.shape)  # (3, 4) ← NumPy calculated 3

# "I want 3 rows — you work out the columns"
matrix6 = returns3.reshape(3, -1)
print(matrix6.shape)  # (3, 4)


#Using ANY negative integer
print(returns3)
matrix7 = returns3.reshape(-2, 2)
print(matrix7)
print(matrix7.shape)  # (3, 4)


(3, 4)
(3, 4)
[ 1  2  3  4  5  6  7  8  9 10 11 12]
[[ 1  2]
 [ 3  4]
 [ 5  6]
 [ 7  8]
 [ 9 10]
 [11 12]]
(6, 2)



💡 In quant work: data.reshape(-1, 1) is extremely common — it converts a 1D array into a column vector, which many ML libraries (scikit-learn) require as input.
<br>
-1 is a 'special signal' to python asking it to 'infer this dimension'
<br>
<br>
You'll see in the output above that python numpy actually accepts ANY -ve interger! BUT -1 is the commonly accepted convention. Anything else will look like a mistake.

**Part 2 flatten() vs ravel() - collapsing it back into 1D**
<br>
*The inverse of reshape — going from 2D → 1D.*

In [51]:
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])

flat1_flatten = matrix.flatten()  # returns a COPY
flat2_ravel = matrix.ravel()    # returns a VIEW (when possible)

print(flat1_flatten)  # [1 2 3 4 5 6]
flat1_flatten[0] = 999
print("Element [0,0] DOESN'T change because it is a copy!: ", matrix[0,0]) 

print(flat2_ravel)  # [1 2 3 4 5 6]
flat2_ravel[0] = 999
print("Element [0,0] DOES change because it is a view!: ", matrix[0,0]) 



[1 2 3 4 5 6]
Element [0,0] DOESN'T change because it is a copy!:  1
[1 2 3 4 5 6]
Element [0,0] DOES change because it is a view!:  999


🔍 flatten() vs ravel() — What's the Difference?
                        flatten()               ravel()
    Returns             A copy                  A view (usually)
    Modifying result    Won't affect original   Will affect original
    Memory              Uses more               Uses less
    Safety              Safer for beginners     More performant (i.e has a duty/is purposefully used)
'''
The difference between a copy and a veiw is that:

                        👁️Copy 📄                               View  
What it is              An entirely new, independent array      A window into the original data
Memory                  Allocates new memory                    Shares the original's memory 
Modifying it            Original untouched ✅                   Changes the original ⚠️  

🏦 Why This Matters in Quant Finance
Imagine you have a clean master array of 5 years of price data. You slice out a section to calculate a rolling metric — if that slice is a view, any manipulation you do accidentally corrupts your source data. That's a silent, hard-to-debug disaster in a trading system.
This is why you'll often see .copy() used explicitly in production quant code:
<br><br>
#Defensive programming — explicitly force a copy<br>
working_data = master_prices[start:end].copy()

**Part 3 Stacking Arrays**
<br>
Now for **combining** arrays. There are three main tools:

np.vstack() — Vertical Stack (stack rows on top of each other)

In [ ]:
# Think: two weeks of returns, want one combined array
week1 = np.array([0.01, -0.02, 0.03])  # Mon-Wed
week2 = np.array([0.00,  0.01, -0.01]) # Mon-Wed

combined = np.vstack([week1, week2])
print(combined)
# [[ 0.01 -0.02  0.03]
#  [ 0.    0.01 -0.01]]
print(combined.shape)  # (2, 3)

[[ 0.01 -0.02  0.03]
 [ 0.    0.01 -0.01]]
(2, 3)


In [62]:
#THIS WONT WORK !! They have to be the same dimensions
week1b = np.array([0.01, -0.02, 0.03])  # Mon-Wed
week2b = np.array([0.00,  0.01]) # Mon-Wed

combinedb = np.vstack([week1b, week2b])
print(combinedb)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 3 and the array at index 1 has size 2

📊 Visualised:
```
week1 →  [0.01  -0.02   0.03]
                  ↕ stack vertically
week2 →  [0.00   0.01  -0.01]

result → [[0.01  -0.02   0.03]
           [0.00   0.01  -0.01]]

np.hstack() — Horizontal Stack (stack columns side by side)

In [ ]:
# Two assets, want them side-by-side
asset_A = np.array([0.01, -0.02, 0.03])
asset_B = np.array([0.02,  0.01, -0.01])
asset_Bb = np.array([[0.02,  0.01, -0.01],[0.02,  0.01, -0.01]]) # <-- You'll see that this one doesn't work

side_by_side = np.hstack([asset_A, asset_B])
print(side_by_side)
# [ 0.01 -0.02  0.03  0.02  0.01 -0.01]
# Note: for 1D arrays, hstack just CONCATENATES

[ 0.01 -0.02  0.03  0.02  0.01 -0.01]


For 2D arrays, hstack adds columns:

In [ ]:
A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5],
              [6]])

B7 = np.array([5]) # <-- You'll see that this one doens't work either

result2 = np.hstack([A, B])
print(result2)
# [[1 2 5]
#  [3 4 6]]
print(result2.shape)  # (2, 3) — gained a column

[[1 2 5]
 [3 4 6]]
(2, 3)


np.concatenate() — The Universal Combiner<br>
vstack and hstack are actually shortcuts. np.concatenate() gives you explicit control via the axis parameter:

In [12]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("")
print("Array A: \n",A)
print("")
print("Array B: \n",B)
print("")

# axis=0 → stack vertically (like vstack)
print(np.concatenate([A, B], axis=0))
# [[1 2]
#  [3 4]
#  [5 6]
#  [7 8]]

print("")
# axis=1 → stack horizontally (like hstack)
print(np.concatenate([A,B], axis=1))
# [[1 2 5 6]
#  [3 4 7 8]]


Array A: 
 [[1 2]
 [3 4]]

Array B: 
 [[5 6]
 [7 8]]

[[1 2]
 [3 4]
 [5 6]
 [7 8]]

[[1 2 5 6]
 [3 4 7 8]]


```

---

## 🎯 The Axis Mental Model

This trips up almost everyone at first, so let's nail it visually:
```
For a 2D array:

        axis=1 →  (across columns)
        ─────────────────────
   ↓  | col0  col1  col2  col3
axis  | col0  col1  col2  col3
 =0   | col0  col1  col2  col3
(down |
 rows)|

axis=0 = down the rows (affects number of rows)
axis=1 = across the columns (affects number of columns)


**Tasks**

In [56]:
# You have 6 months of returns for 1 asset as a flat array
monthly_returns = np.array([0.02, -0.01, 0.03, 0.00, -0.02, 0.04])
print(monthly_returns)

[ 0.02 -0.01  0.03  0.   -0.02  0.04]


In [ ]:
# Task 1: Reshape into 2 quarters × 3 months
monthly_returns_reshaped = monthly_returns.reshape(2,3)
print(monthly_returns_reshaped)


print("")
# Task 2: Add a second asset's returns and vstack them
asset_B_returns = np.array([0.01, 0.02, -0.01, 0.03, 0.01, -0.02])
assets_combined = np.vstack([monthly_returns, asset_B_returns])
print("Using vstack: \n",assets_combined)


print("")
# Task 3: Use concatenate with axis=0 to do the same as vstack
assets_combined_concat = np.concatenate([monthly_returns, asset_B_returns], axis=0)
print("Using .concatenate and .reshape: \n",assets_combined_concat.reshape(2,-1))


print("")
# Task 4: Flatten your result back to 1D using both flatten() and ravel()
#          — then modify element [0] of each and check if the original changed
monthly_returns_flat = monthly_returns.flatten()
monthly_returns_flat[0] = 999
print(monthly_returns[0])
if(monthly_returns[0] != 0.02): 
    print("Changed")
else:
    print("Unchanged")


[[ 0.02 -0.01  0.03]
 [ 0.   -0.02  0.04]]

Using vstack: 
 [[ 0.02 -0.01  0.03  0.   -0.02  0.04]
 [ 0.01  0.02 -0.01  0.03  0.01 -0.02]]

Using .concatenate and .reshape: 
 [[ 0.02 -0.01  0.03  0.   -0.02  0.04]
 [ 0.01  0.02 -0.01  0.03  0.01 -0.02]]
0.02
Unchanged


In [55]:
monthly_returns_ravel = monthly_returns.ravel()
monthly_returns_ravel[0] = 999 #This should change the original
print(monthly_returns[0])
if(monthly_returns[0] != 0.02): 
    print("Changed")
else:
    print("Unchanged")

999.0
Changed


**TASK 3: EXTRA NOTES ON CONCAT**

This is actually a subtle but important distinction. For 1D arrays, concatenate(axis=0) and vstack do NOT produce the same result.

concatenate with axis=0 **only** behaves like vstack **when your inputs are already 2D**: (see code below)

In [69]:
#Re: Task 3: Use concatenate with axis=0 to do the same as vstack

# vstack on 1D arrays → 2D array (2, 6)
print(np.vstack([monthly_returns, asset_B_returns]))
# [[ 0.02 -0.01  0.03  0.00 -0.02  0.04]
#  [ 0.01  0.02 -0.01  0.03  0.01 -0.02]]

print("")
# concatenate axis=0 on 1D arrays → still 1D! (12,)
print(np.concatenate([monthly_returns, asset_B_returns], axis=0)) # <- this was my original answer
# [ 0.02 -0.01  0.03  0.00 -0.02  0.04  0.01  0.02 -0.01  0.03  0.01 -0.02]

[[ 0.02 -0.01  0.03  0.   -0.02  0.04]
 [ 0.01  0.02 -0.01  0.03  0.01 -0.02]]

[ 0.02 -0.01  0.03  0.   -0.02  0.04  0.01  0.02 -0.01  0.03  0.01 -0.02]


In [74]:
#the correct answer to 3
# Reshape inputs to 2D first, THEN concatenate
print(monthly_returns)
print("Reshaped to 2D: ", monthly_returns.reshape(1, -1))
print("")
print(asset_B_returns)
print("Reshaped to 2D: ", asset_B_returns.reshape(1, -1))

print("")
assets_combined_concat = np.concatenate([
    monthly_returns.reshape(1, -1),
    asset_B_returns.reshape(1, -1)
], axis=0)

print(assets_combined_concat)
# [[ 0.02 -0.01  0.03  0.00 -0.02  0.04]
#  [ 0.01  0.02 -0.01  0.03  0.01 -0.02]]

[ 0.02 -0.01  0.03  0.   -0.02  0.04]
Reshaped to 2D:  [[ 0.02 -0.01  0.03  0.   -0.02  0.04]]

[ 0.01  0.02 -0.01  0.03  0.01 -0.02]
Reshaped to 2D:  [[ 0.01  0.02 -0.01  0.03  0.01 -0.02]]

[[ 0.02 -0.01  0.03  0.   -0.02  0.04]
 [ 0.01  0.02 -0.01  0.03  0.01 -0.02]]
